# Moonshine-base → banBERT 50k Transplant + Fine-tune on 1.4M Unified Corpus

**Flow:**
```
Sanji27/fountain_base_21ep
  → banBERT 50k vocab transplant  (Cell 1)
  → fine-tune on 1.4M unified corpus  (Cell 2+)
```

**Key decisions vs old notebooks:**
- No Stage 1 (vanilla pre-fine-tune) — 1.4M samples is sufficient signal
- LR = 5e-5 (middle ground: decoder is randomly initialized but data volume is large)
- warmup_steps = 2000 (longer than before to protect encoder early on)
- WER_SUBSET = 256 (more reliable signal than 64)
- MIN_AUDIO_SEC = 1.0 (matches data prep)
- NUM_WORKERS = 4 (Linux)
- 3-tier checkpointing: best/ + latest/ + epoch_N/ every 5 epochs
- CSV input (audio_path, text, duration, is_augmented)
- Full training log written to training_log.csv
```

## Cell 1 — Tokenizer Transplant

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSpeechSeq2Seq, BertTokenizer

# ─── CONFIG ───────────────────────────────────────────────────────────────────
BASE_MODEL_ID    = "Sanji27/fountain_base_21ep"
NEW_TOKENIZER_ID = "banglagov/banBERT-Base"
SAVE_DIR         = "/home/serima/Sanjidh090/unified_corpus/fountain_banbert_transplant"
HF_TOKEN         = "hf_"   # fill in

print(f"Loading base model: {BASE_MODEL_ID}")
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

print(f"Loading new tokenizer: {NEW_TOKENIZER_ID}")
new_tokenizer = BertTokenizer.from_pretrained(NEW_TOKENIZER_ID)

old_vocab = model.config.vocab_size
new_vocab = len(new_tokenizer)
print(f"Old vocab: {old_vocab:,}  →  New vocab: {new_vocab:,}")

# ─── SURGERY ──────────────────────────────────────────────────────────────────
print("Resizing embeddings...")
model.resize_token_embeddings(new_vocab)

# banBERT is BERT-style: CLS=BOS, SEP=EOS, PAD=PAD
model.config.vocab_size             = new_vocab
model.config.pad_token_id           = new_tokenizer.pad_token_id       # 0
model.config.bos_token_id           = new_tokenizer.cls_token_id       # 101
model.config.eos_token_id           = new_tokenizer.sep_token_id       # 102
model.config.decoder_start_token_id = new_tokenizer.cls_token_id       # 101

# clear any stale generation limits from moonshine config
model.generation_config.max_length       = None
model.generation_config.forced_bos_token_id = None

print(f"Token IDs — BOS:{model.config.bos_token_id}  "
      f"EOS:{model.config.eos_token_id}  "
      f"PAD:{model.config.pad_token_id}")

# ─── VERIFY ROUND-TRIP ────────────────────────────────────────────────────────
test_sentences = [
    "আমার সোনার বাংলা",
    "মানসিক রোগ সম্পর্কে সচেতন হওয়া জরুরি",
    "সে বাজারে গিয়েছিল",
]
for s in test_sentences:
    ids  = new_tokenizer.encode(s, add_special_tokens=False)
    back = new_tokenizer.decode(ids, skip_special_tokens=True)
    status = "✓" if back.strip() == s.strip() else "✗"
    print(f"   {status} [{len(ids)} tokens] {s[:40]}")

# ─── SAVE ─────────────────────────────────────────────────────────────────────
import os
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
new_tokenizer.save_pretrained(SAVE_DIR)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\n✅ Transplant complete → {SAVE_DIR}")
print(f"   Params: {total_params:.1f}M")
print(f"   Vocab:  {new_vocab:,} tokens")

## Cell 2 — Training (train_base_banbert.py)

In [ ]:
"""
train_base_banbert.py
=====================
Bengali Moonshine BASE — banBERT 50k edition
  Encoder  : moonshine-base (cold, no Stage 1)
  Tokenizer: banglagov/banBERT-Base  (50k WordPiece)
  Data     : 1.4M unified corpus (train/val/test CSV)
  LR       : 5e-5  (mid-range: decoder is random init but data volume is large)
  Warmup   : 2000 steps  (protects encoder early on)
  Checkpoints: best/ + latest/ + epoch_N/ every 5 epochs
  Logging  : console + training_log.csv
"""

import os, time, random, csv as csv_module
import numpy as np
import librosa
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSpeechSeq2Seq
import schedulefree

# ═══════════════════════════════════════════════════════════════════════════════
#  PATHS
# ═══════════════════════════════════════════════════════════════════════════════
CORPUS_ROOT = Path("/home/serima/Sanjidh090/unified_corpus")

TRAIN_CSV   = CORPUS_ROOT / "train.csv"
VAL_CSV     = CORPUS_ROOT / "val.csv"
TEST_CSV    = CORPUS_ROOT / "test.csv"

HF_MODEL    = str(CORPUS_ROOT / "fountain_banbert_transplant")
HF_TOKEN    = "hf_"   # fill in

WORK_DIR    = CORPUS_ROOT / "runs" / "banbert_unified"
CKPT_DIR    = WORK_DIR / "checkpoints"
LOG_CSV     = WORK_DIR / "training_log.csv"

for d in [WORK_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
SAMPLE_RATE    = 16_000
MAX_AUDIO_SEC  = 30.0
MIN_AUDIO_SEC  = 1.0      # matches data prep (was 4.0 in old notebooks)
MAX_TOKENS     = 190      # banBERT ~1-to-1 fertility, 190 is safe

BATCH_SIZE     = 4
GRAD_ACCUM     = 8        # effective batch = 32
EPOCHS         = 30       # long run — early stopping will terminate
LR             = 5e-5     # mid-range for cold encoder + random decoder
WARMUP_STEPS   = 2000     # longer warmup to protect encoder
LOG_EVERY      = 100      # steps
PATIENCE       = 5        # epochs without WER improvement
WER_SUBSET     = 256      # val samples for WER per epoch
SNAPSHOT_EVERY = 5        # save epoch_N/ snapshot every N epochs

NUM_WORKERS    = 4        # Linux

RESUME         = True
RESUME_DIR     = CKPT_DIR / "latest"

# ─── device ───────────────────────────────────────────────────────────────────
device   = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device == "cuda" and not use_bf16
dtype    = torch.bfloat16 if use_bf16 else torch.float16 if use_fp16 else torch.float32

print("=" * 65)
print(f"  Model    : {HF_MODEL}")
print(f"  Device   : {device}  |  dtype: {dtype}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {props.total_memory / 1e9:.1f} GB")
print(f"  Eff.batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR       : {LR}  |  warmup: {WARMUP_STEPS} steps")
print(f"  Epochs   : {EPOCHS}  |  patience: {PATIENCE}")
print(f"  Resume   : {RESUME}")
print("=" * 65)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Load tokenizer + model
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[1/4] Loading model ...")

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)
print(f"   ✓ Tokenizer vocab: {tokenizer.vocab_size:,}")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    HF_MODEL, trust_remote_code=True, torch_dtype=torch.float32
)

BOS_ID = model.config.decoder_start_token_id or tokenizer.cls_token_id
EOS_ID = model.config.eos_token_id            or tokenizer.sep_token_id
PAD_ID = model.config.pad_token_id            or tokenizer.pad_token_id or 0

print(f"   ✓ Token IDs — BOS:{BOS_ID}  EOS:{EOS_ID}  PAD:{PAD_ID}")

model.generation_config.max_length = None
model.gradient_checkpointing_enable()

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"   ✓ {total:.1f}M params  ({trainable:.1f}M trainable)")

model = model.to(device)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — Dataset
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[2/4] Building dataloaders ...")


class BengaliASRDataset(Dataset):
    def __init__(self, csv_path, bos_id, eos_id, label):
        self.bos_id = bos_id
        self.eos_id = eos_id
        df = pd.read_csv(csv_path)
        # only need audio_path and text
        self.samples = list(zip(df["audio_path"].tolist(), df["text"].tolist()))
        hrs = df["duration"].sum() / 3600 if "duration" in df.columns else 0
        print(f"   ✓ {label:<8} {len(self.samples):>8,} samples  "
              f"({hrs:.1f}h)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        try:
            audio_path, transcript = self.samples[idx]
            transcript = str(transcript).strip()
            if not transcript:
                return None

            audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
            audio    = audio.astype(np.float32)

            dur = len(audio) / SAMPLE_RATE
            if not (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC):
                return None

            # pad to multiple of 160
            remainder = len(audio) % 160
            if remainder:
                audio = np.concatenate(
                    [audio, np.zeros(160 - remainder, dtype=np.float32)]
                )

            ids = tokenizer.encode(transcript, add_special_tokens=False)
            if len(ids) == 0 or len(ids) > MAX_TOKENS - 2:
                return None

            ids = [self.bos_id] + ids + [self.eos_id]
            return {
                "audio":     torch.tensor(audio, dtype=torch.float32),
                "input_ids": torch.tensor(ids,   dtype=torch.long),
            }
        except Exception:
            return None


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    max_a      = ((max(b["audio"].shape[0] for b in batch) + 159) // 160) * 160
    audio_b    = torch.zeros(len(batch), max_a)
    audio_mask = torch.zeros(len(batch), max_a, dtype=torch.long)

    for i, b in enumerate(batch):
        L = b["audio"].shape[0]
        audio_b[i, :L]    = b["audio"]
        audio_mask[i, :L] = 1

    max_t   = max(b["input_ids"].shape[0] for b in batch)
    token_b = torch.full((len(batch), max_t), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = b["input_ids"].shape[0]
        token_b[i, :L] = b["input_ids"]

    return {"audio": audio_b, "audio_mask": audio_mask, "input_ids": token_b}


train_ds = BengaliASRDataset(TRAIN_CSV, BOS_ID, EOS_ID, "train")
val_ds   = BengaliASRDataset(VAL_CSV,   BOS_ID, EOS_ID, "val")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda"), prefetch_factor=2,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
)

steps_per_epoch = (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM
print(f"   ✓ Train batches: {len(train_loader):,}  ({steps_per_epoch:,} grad steps/epoch)")
print(f"   ✓ Val   batches: {len(val_loader):,}")


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — Optimizer + Scaler
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[3/4] Initializing optimizer ...")

optimizer = schedulefree.AdamWScheduleFree(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    warmup_steps=WARMUP_STEPS,
)

scaler    = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
amp_dtype = dtype if device == "cuda" else torch.float32

# ─── state ────────────────────────────────────────────────────────────────────
start_epoch = 1
BEST_WER    = float("inf")
BEST_VAL    = float("inf")
patience_ct = 0

# ─── resume ───────────────────────────────────────────────────────────────────
if RESUME and (RESUME_DIR / "training_state.pt").exists():
    print(f"   Resuming from: {RESUME_DIR}")
    state = torch.load(RESUME_DIR / "training_state.pt", map_location=device)
    model.load_state_dict(
        torch.load(RESUME_DIR / "model_state.pt", map_location=device)
    )
    optimizer.load_state_dict(state["optimizer"])
    for pg in optimizer.param_groups:
        pg["lr"] = LR
    start_epoch = state["epoch"] + 1
    BEST_WER    = state.get("best_wer", float("inf"))
    BEST_VAL    = state.get("best_val", float("inf"))
    patience_ct = state.get("patience", 0)
    print(f"   ✓ Resumed  epoch={state['epoch']}  "
          f"best_WER={BEST_WER*100:.1f}%  patience={patience_ct}/{PATIENCE}")
else:
    print("   Starting from scratch ...")

# ─── init log csv ─────────────────────────────────────────────────────────────
if not LOG_CSV.exists():
    with open(LOG_CSV, "w", newline="") as f:
        writer = csv_module.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "wer", "time_min", "saved"])


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ═══════════════════════════════════════════════════════════════════════════════
def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for ac in a:
        p, d[0] = d[:], d[0] + 1
        for j, bc in enumerate(b):
            d[j+1] = min(p[j] + (ac != bc), d[j] + 1, p[j+1] + 1)
    return d[len(b)]


def corpus_wer(hyps, refs):
    total_w = total_e = 0
    for h, r in zip(hyps, refs):
        hw, rw = h.split(), r.split()
        total_w += len(rw)
        total_e += edit_distance(hw, rw)
    return total_e / max(1, total_w)


def save_checkpoint(directory, epoch, val_loss, wer, is_best=False):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    optimizer.eval()
    model.save_pretrained(directory)
    tokenizer.save_pretrained(directory)
    torch.save(model.state_dict(), directory / "model_state.pt")
    torch.save({
        "epoch":     epoch,
        "val_loss":  val_loss,
        "wer":       wer,
        "best_wer":  BEST_WER,
        "best_val":  BEST_VAL,
        "patience":  patience_ct,
        "optimizer": optimizer.state_dict(),
    }, directory / "training_state.pt")
    optimizer.train()
    tag = " [BEST]" if is_best else ""
    print(f"   ✅ Saved{tag} → {directory.name}  "
          f"epoch={epoch}  val={val_loss:.4f}  WER={wer*100:.1f}%")


# ═══════════════════════════════════════════════════════════════════════════════
#  TRAIN EPOCH
# ═══════════════════════════════════════════════════════════════════════════════
def train_epoch(epoch):
    optimizer.train()
    model.train()
    total_loss = 0.0
    t0         = time.time()
    optimizer.zero_grad()
    steps      = 0

    for step, batch in enumerate(train_loader):
        if batch is None:
            continue

        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)

        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype,
                            enabled=(device == "cuda")):
            out = model(
                input_values=audio,
                attention_mask=audio_mask,
                decoder_input_ids=dec_input,
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        is_accum_step = (step + 1) % GRAD_ACCUM == 0
        is_last_step  = (step + 1) == len(train_loader)

        if is_accum_step or is_last_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            steps += 1

            if steps % LOG_EVERY == 0:
                avg       = total_loss / (step + 1)
                elapsed   = time.time() - t0
                remaining = steps_per_epoch - steps
                eta       = elapsed / steps * remaining if steps > 0 else 0
                print(f"   ep{epoch} [{steps:>5}/{steps_per_epoch}]  "
                      f"loss={avg:.4f}  ETA={eta/60:.0f}min")

    return total_loss / max(1, len(train_loader))


# ═══════════════════════════════════════════════════════════════════════════════
#  VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def validate():
    optimizer.eval()
    model.eval()
    total_loss = 0.0
    n          = 0
    hyps, refs = [], []

    for batch in val_loader:
        if batch is None:
            continue

        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)

        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype,
                            enabled=(device == "cuda")):
            out = model(
                input_values=audio,
                attention_mask=audio_mask,
                decoder_input_ids=dec_input,
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
        total_loss += loss.item()
        n += 1

        # WER on random subset (val_loader is shuffled)
        if len(hyps) < WER_SUBSET:
            generated_ids = model.generate(
                inputs=audio,
                attention_mask=audio_mask,
                max_new_tokens=MAX_TOKENS,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                decoder_start_token_id=BOS_ID,
                num_beams=4,
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,
            )
            for g_ids, ref_ids in zip(generated_ids, input_ids):
                ref_ids = ref_ids[ref_ids != -100].tolist()
                hyps.append(tokenizer.decode(g_ids.tolist(),  skip_special_tokens=True))
                refs.append(tokenizer.decode(ref_ids,         skip_special_tokens=True))

    return total_loss / max(1, n), corpus_wer(hyps, refs)


# ═══════════════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n[4/4] Training ...")
print(f"{'='*65}")
print(f"  Epoch range : {start_epoch} → {EPOCHS}")
print(f"  Grad steps  : {steps_per_epoch:,} / epoch")
print(f"{'='*65}\n")

for epoch in range(start_epoch, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(epoch)
    val_loss, wer = validate()
    epoch_min  = (time.time() - t0) / 60

    print(f"\nepoch {epoch}/{EPOCHS}  "
          f"train={train_loss:.4f}  val={val_loss:.4f}  "
          f"WER={wer*100:.1f}%  time={epoch_min:.0f}min")

    saved = False

    # ── always save latest (for resume) ───────────────────────────────────────
    save_checkpoint(CKPT_DIR / "latest", epoch, val_loss, wer)

    # ── save best if WER improved ──────────────────────────────────────────────
    if wer < BEST_WER:
        BEST_WER    = wer
        BEST_VAL    = val_loss
        patience_ct = 0
        save_checkpoint(CKPT_DIR / "best", epoch, val_loss, wer, is_best=True)
        saved = True
    else:
        patience_ct += 1
        print(f"   ⚠  No WER improvement — "
              f"patience {patience_ct}/{PATIENCE}  "
              f"(best so far: {BEST_WER*100:.1f}%)")

    # ── periodic snapshot every N epochs ──────────────────────────────────────
    if epoch % SNAPSHOT_EVERY == 0:
        snap_dir = CKPT_DIR / f"epoch_{epoch:03d}"
        save_checkpoint(snap_dir, epoch, val_loss, wer)

    # ── write log row ──────────────────────────────────────────────────────────
    with open(LOG_CSV, "a", newline="") as f:
        csv_module.writer(f).writerow([
            epoch, f"{train_loss:.4f}", f"{val_loss:.4f}",
            f"{wer*100:.2f}", f"{epoch_min:.1f}",
            "yes" if saved else "no"
        ])

    print()

    if patience_ct >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch} "
              f"(best WER: {BEST_WER*100:.1f}%)")
        break


print(f"\n{'='*65}")
print(f"  Training complete")
print(f"  Best WER : {BEST_WER*100:.1f}%")
print(f"  Best val : {BEST_VAL:.4f}")
print(f"  Log      : {LOG_CSV}")
print(f"  Best ckpt: {CKPT_DIR / 'best'}")
print(f"{'='*65}")